# Gradient and Hessian of simple machines

This notebook accompanies Exercises 2.1 and 2.2 of the course. Mathematical calculations take place in the course's matrix and vector spaces; JAX then checks the gradient and, particularly, the action of the Hessian on a direction.

We shall not form a large Hessian matrix. The `jax.jvp` transformation directly computes
$$
\operatorname{Hess}\mathcal L(p)\,\delta p
=D(\nabla\mathcal L)(p)[\delta p].
$$
The second derivative $D^2\mathcal L(p)$ is a bilinear form; $\operatorname{Hess}\mathcal L(p)$ is its associated operator under the inner product. Similarly, for a matrix variable $A$, $\nabla_A\mathcal L$ denotes the Riesz representative of $D_A\mathcal L$ for the Frobenius inner product:
$$
D_A\mathcal L(A,b)[\delta A]
=\langle\nabla_A\mathcal L(A,b),\delta A\rangle_F.
$$

## Exercises

1. [Exercise 2.1 — Hessian of linear regression](#exercice-2-1)
2. [Exercise 2.2 — Gradient and Hessian of a perceptron](#exercice-2-2)

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt

print(f"JAX {jax.__version__}")

<a id="exercice-2-1"></a>
## Exercise 2.1 — Hessian of linear regression

For $A\in\mathbb R^{m\times n}$ and $b\in\mathbb R^m$, consider
$$
\mathcal L(A,b)=\frac12\sum_{(x,z)\in\mathcal D}|Ax+b-z|^2.
$$

Show that the Hessian is independent of $(A,b)$ and its action on $(\delta A,\delta b)$ is
$$
\operatorname{Hess}\mathcal L(A,b)(\delta A,\delta b)
=\left(
\sum_{(x,z)\in\mathcal D}(\delta Ax+\delta b)x^\top,
\sum_{(x,z)\in\mathcal D}(\delta Ax+\delta b)
\right).
$$

Then establish
$$
\left\langle\operatorname{Hess}\mathcal L(A,b)(\delta A,\delta b),
(\delta A,\delta b)\right\rangle
=\sum_{(x,z)\in\mathcal D}|\delta Ax+\delta b|^2
$$
and deduce the convexity of $\mathcal L$.

### Verification with JAX

The data are stored row-wise: `x` has shape `(n_data, n)`, so the machine computes `x @ A.T + b`. The parameters and their perturbation are dictionaries with the same structure.

In [ ]:
cle = jax.random.key(2026)
cle_x, cle_z, cle_A, cle_b, cle_dA, cle_db = jax.random.split(cle, 6)
n_data, n, m = 8, 3, 2

x = jax.random.normal(cle_x, (n_data, n))
z = jax.random.normal(cle_z, (n_data, m))
parametres = {
    "A": jax.random.normal(cle_A, (m, n)),
    "b": jax.random.normal(cle_b, (m,)),
}
direction = {
    "A": jax.random.normal(cle_dA, (m, n)),
    "b": jax.random.normal(cle_db, (m,)),
}

def cout_lineaire(parametres, x, z):
    residu = x @ parametres["A"].T + parametres["b"] - z
    return 0.5 * jnp.sum(residu**2)

Use JAX to compute the Hessian action on `direction`, then construct the same action from the course formula. Finally, verify the quadratic-form formula.

In [ ]:
# Complete this section.

### Further question

Non-negativity proves convexity, but not always strict convexity. Construct a dataset for which a non-zero direction satisfies $\delta Ax+\delta b=0$ for every observed input. What then happens to the quadratic form?

In [ ]:
# Complete this section.

<a id="exercice-2-2"></a>
## Exercise 2.2 — Gradient and Hessian of a perceptron

Consider
$$
\Phi(x,W,b)=\sigma(Wx+b)
$$
and
$$
\mathcal L(W,b)=\frac12\sum_{(x,z)\in\mathcal D}
|\sigma(Wx+b)-z|^2.
$$

With $a_x=Wx+b$ and $r_{x,z}=\sigma(a_x)-z$, establish the gradient formulae given in the course. For a direction $(\delta W,\delta b)$, set
$$
v_x=\delta Wx+\delta b,
\qquad
\gamma_{x,z}=\sigma'(a_x)\odot\sigma'(a_x)
+r_{x,z}\odot\sigma''(a_x),
$$
then establish
$$
\operatorname{Hess}\mathcal L(W,b)(\delta W,\delta b)
=\left(
\sum (\gamma_{x,z}\odot v_x)x^\top,
\sum \gamma_{x,z}\odot v_x
\right).
$$

### Verification for $\sigma=\tanh$

For `tanh`,
$$
\sigma'(t)=1-\tanh^2(t),
\qquad
\sigma''(t)=-2\tanh(t)(1-\tanh^2(t)).
$$

Implement the gradient and Hessian-action formulae, then compare them with JAX results.

In [ ]:
parametres_perceptron = {
    "W": jax.random.normal(cle_A, (m, n)),
    "b": jax.random.normal(cle_b, (m,)),
}
direction_perceptron = {
    "W": jax.random.normal(cle_dA, (m, n)),
    "b": jax.random.normal(cle_db, (m,)),
}

def cout_perceptron(parametres, x, z):
    a = x @ parametres["W"].T + parametres["b"]
    residu = jnp.tanh(a) - z
    return 0.5 * jnp.sum(residu**2)

In [ ]:
# Complete this section.

### Non-convexity

In the scalar case $x=z=0$ and $\sigma=\tanh$, the loss depends only on $b$:
$$
\mathcal L(b)=\frac12\tanh^2(b).
$$

Compute its second derivative and find a region where it is negative.

In [ ]:
# Complete this section.

## Summary

For linear regression, the Hessian is positive semidefinite and independent of the parameters. For a perceptron, the term containing the residual and $\sigma''$ can make the Hessian indefinite: the objective is therefore not necessarily convex.

In both cases, `jax.jvp(jax.grad(...))` directly computes the Hessian action. This representation is more natural and often much cheaper than constructing the full Hessian matrix.